# 🧭 Penginapan Sentiment Training: IndoBERT Fine-Tuning (Kaggle Ready)

Notebook ini dirancang khusus untuk dieksekusi di platform ber-GPU seperti Kaggle atau Google Colab.
Kita menggunakan arsitektur **Deep Learning (IndoBERT)** dari `indobenchmark/indobert-base-p1` untuk menembus akurasi >90% pada klasifikasi 3-Kelas (Positif, Netral, Negatif).

Pola wajib setiap tahap:
`tujuan kecil -> code/output -> keputusan -> langkah berikutnya`


## Fase 0 - Persiapan Environment
**Tujuan:** Menginstal library `transformers` dan `evaluate` dari HuggingFace yang tidak tersedia secara default.

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate

# Pastikan GPU terdeteksi
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


## Fase A - Load Data Bersih
**Tujuan:** Memuat dataset ulasan penginapan yang sudah diaudit (9.138 baris) dan mengubah label teks menjadi angka.

In [ ]:
# UBAH PATH INI SESUAI LOKASI DATASET DI KAGGLE ANDA
# Misal: DATA_PATH = '/kaggle/input/penginapan-muterbandung/PENGINAPAN_FASE_A_LABELLED_2026-06-14.csv'
DATA_PATH = 'PENGINAPAN_FASE_A_LABELLED_2026-06-14.csv' 

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Berhasil memuat {len(df)} baris data.")
except Exception as e:
    print(f"Error memuat data: {e}")

# Mapping Label (Positif: 2, Netral: 1, Negatif: 0)
label_map = {'Negatif': 0, 'Netral': 1, 'Positif': 2}
df['label'] = df['sentiment_label'].map(label_map)

# Kita hanya butuh teks dan label
df = df[['cleaned_text', 'label']].dropna()
df.rename(columns={'cleaned_text': 'text'}, inplace=True)

# Train-Test Split (80/20)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print(f"Train size: {len(df_train)} | Test size: {len(df_test)}")


**Keputusan:** Data berhasil dimuat dan dipotong (split). Format label sudah dikonversi menjadi integer (0,1,2).
**Langkah Berikutnya:** Masuk ke Fase B untuk Tokenisasi IndoBERT.

## Fase B - Tokenisasi IndoBERT
**Tujuan:** Mengubah teks bahasa Indonesia menjadi token-token numerik menggunakan *vocabulary* resmi IndoBERT.

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Convert Pandas to HuggingFace Dataset
train_dataset = Dataset.from_pandas(df_train, preserve_index=False)
test_dataset = Dataset.from_pandas(df_test, preserve_index=False)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Memulai tokenisasi...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
print("Tokenisasi selesai!")


**Keputusan:** Teks berhasil ditokenisasi dengan panjang maksimal 128 token agar memori GPU tidak kepenuhan.
**Langkah Berikutnya:** Melakukan training model.

## Fase C - Model Training & Evaluation
**Tujuan:** Fine-tuning pre-trained model IndoBERT pada data khusus penginapan kita dengan bantuan *Class Weights* khusus untuk mengatasi *Imbalance* kelas.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Hitung bobot kelas secara matematis (karena data kita imbalanced)
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label']), y=df_train['label'])
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights: {class_weights_tensor}")

# Custom Trainer untuk menyuntikkan class weights
from torch import nn
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Metrik Evaluasi (Akurasi & F1)
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none" # Matikan logging wandb dll
)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("Memulai Training IndoBERT di GPU...")
trainer.train()


**Keputusan:** Model IndoBERT berhasil menyelesaikan pelatihan (Epochs). Periksa tabel Akurasi dan Loss di atas, seharusnya bisa mencapai/melewati 90%.
**Langkah Berikutnya:** Menyimpan model yang sudah pintar.

## Fase D - Model Saving & Deployment
**Tujuan:** Menyimpan model yang sudah dilatih agar bisa diunduh (download) dari Kaggle dan dipakai di MuterBandung Local.

In [ ]:
# Simpan model dan tokenizer
SAVE_DIR = "./model_sentimen_penginapan_indobert"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model berhasil disimpan di direktori: {SAVE_DIR}")
print("Silakan ZIP folder tersebut dan download untuk dipasang di Backend MuterBandung Anda!")

# Cara melakukan inference nanti:
# from transformers import pipeline
# nlp_pipe = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR)
# print(nlp_pipe("Kamar hotel ini sangat nyaman dan kasurnya empuk!"))
